In [19]:
import numpy as np
import os
import shutil
import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix

In [20]:
def clean_logs(data_dir):
    logs_dir = os.path.join(data_dir, "logs")
    shutil.rmtree(logs_dir, ignore_errors=True)
    return logs_dir

In [21]:
import tensorflow as tf
import zipfile

def download_and_read(url):
    zip_path = tf.keras.utils.get_file(
        fname="sentiment_labelled_sentences.zip",
        origin=url,
        extract=False,
        cache_dir="."
    )

    labeled_sentences = []

    with zipfile.ZipFile(zip_path, "r") as archive:
        for filename in archive.namelist():
            if not filename.endswith("_labelled.txt"):
                continue

            text = archive.read(filename).decode("utf-8")

            # Gunakan newline biasa agar kalimat tidak terpotong
            # oleh karakter pemisah Unicode.
            for line_number, line in enumerate(text.split("\n"), start=1):
                line = line.strip()

                if not line:
                    continue

                parts = line.rsplit("\t", 1)

                if len(parts) != 2:
                    raise ValueError(
                        f"Format data tidak valid di {filename}, "
                        f"baris {line_number}: {line!r}"
                    )

                sentence, label = parts

                if label not in ("0", "1"):
                    raise ValueError(
                        f"Label tidak valid di {filename}, "
                        f"baris {line_number}: {label!r}"
                    )

                labeled_sentences.append((sentence, label))

    return labeled_sentences

In [22]:
class SentimentAnalysisModel(tf.keras.Model):
    def __init__(self, vocab_size, max_seqlen, **kwargs):
        super(SentimentAnalysisModel, self).__init__(**kwargs)
        self.embedding = tf.keras.layers.Embedding(vocab_size, max_seqlen)
        self.bilstm = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(max_seqlen)
        )
        self.dense = tf.keras.layers.Dense(64, activation="relu")
        self.out = tf.keras.layers.Dense(1, activation="sigmoid")

    def call(self, x):
        x = self.embedding(x)
        x = self.bilstm(x)
        x = self.dense(x)
        x = self.out(x)
        return x

In [23]:
# set random seed
tf.random.set_seed(42)

In [24]:
# clean up log area
data_dir = "./data"
logs_dir = clean_logs(data_dir)

In [26]:
# Download dan baca dataset
labeled_sentences = download_and_read(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00331/sentiment%20labelled%20sentences.zip"
)

# Pisahkan kalimat dan label
sentences = [sentence for sentence, label in labeled_sentences]
labels = [int(label) for sentence, label in labeled_sentences]

# Periksa hasil pembacaan
print("Jumlah kalimat:", len(sentences))
print("Jumlah label:", len(labels))
print("Contoh kalimat:", sentences[0])
print("Contoh label:", labels[0])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa2 in position 37: invalid start byte

In [ ]:
# tokenize sentences
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_counts)
print("vocabulary size: {:d}".format(vocab_size))